# Autonomic multi LLM system coordinierer

Ziel ist ein autonomes System, das drei kleine Sprachmodelle parallel laufen lässt. Ihre Ergebnisse werden bewertet, um die präziseste Aussage zu priorisieren und mit wertvollen Informationen der anderen Modelle anzureichern.

In [1]:
# import
import asyncio
from typing import Any, Dict, List, Set

import nltk
import numpy as np
import ollama
import pandas as pd
import psutil
from pathlib import Path

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
from nltk.corpus import stopwords

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

In [2]:
# Globals
_cfg: Dict[str, Any] = globals().setdefault("_AGENT_CONFIG", {})

# Netzwerk- und Ausführungs-Einstellungen für Ollama
OLLAMA_HOST: str        = _cfg.setdefault("OLLAMA_HOST", "http://127.0.0.1:11434")
SOCKET_TIMEOUT: float   = _cfg.setdefault("SOCKET_TIMEOUT", 1.0)
API_TIMEOUT: float      = _cfg.setdefault("API_TIMEOUT", 5.0)
MAX_START_RETRIES: int  = _cfg.setdefault("MAX_START_RETRIES", 6)
RETRY_WAIT_SEC: float   = _cfg.setdefault("RETRY_WAIT_SEC", 1.5)

OLAMA_MODEL_CODE_LOW    = _cfg.setdefault("OLAMA_MODEL_CODE_LOW",    "codestral:22b")
OLAMA_MODEL_CODE_HIGH   = _cfg.setdefault("OLAMA_MODEL_CODE_HIGH",   "qwen3-coder-next-ud-q4-k-xl")
OLAMA_MODEL_LOGIC_LOW   = _cfg.setdefault("OLAMA_MODEL_LOGIC_LOW",   "deepseek-r1:8b")
OLAMA_MODEL_LOGIC_HIGH  = _cfg.setdefault("OLAMA_MODEL_LOGIC_HIGH",  "qwen3-coder-next-ud-q4-k-xl")
OLAMA_MODEL_TEXT_LOW    = _cfg.setdefault("OLAMA_MODEL_TEXT_LOW",    "gemma3:4b")
OLAMA_MODEL_TEXT_HIGH   = _cfg.setdefault("OLAMA_MODEL_TEXT_HIGH",   "qwen3.6:27b")


In [3]:
# def for run
def clean_query(query: str, 
                language: str = 'german', 
                additional_stopwords: List[str] = None, 
                exclude_stopwords: List[str] = None) -> str:
    """
    Bereinigt die Benutzeranfrage basierend auf einer Standard-Bibliothek (NLTK),
    erlaubt das Hinzufügen eigener Stopwörter und das Ausschließen 
    bestimmter Wörter von der Löschung (Exclude / Whitelist).
    """
    print('clean_query')
    if additional_stopwords is None:
        additional_stopwords = []
    if exclude_stopwords is None:
        exclude_stopwords = []
    stop_words: Set[str] = set(stopwords.words(language))
    stop_words.update(map(str.lower, additional_stopwords))
    for word in exclude_stopwords:
        stop_words.discard(word.lower())
    words = query.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return " ".join(filtered_words)

def get_available_local_models(project_models_dir: str = "../data/models") -> dict:
    """
    Ruft alle aktiv bei Ollama registrierten Modelle direkt über die API ab,
    nutzt dabei die globalen Host-Einstellungen, ermittelt deren Dateigröße 
    und gibt ein Dictionary zurück ({name: größe_gb}).
    Prüft zusätzlich das lokale Projektverzeichnis (data/models) auf .gguf-Dateien.
    """
    print('get_available_local_models')
    local_models_dict = {}
    try:
        response = ollama.list()
        models_info = response.get('models', []) if isinstance(response, dict) else getattr(response, 'models', [])
        print(f"\n Registrierte Ollama-Modelle (über API: {OLLAMA_HOST}) ")
        if not models_info:
            print("  Keine Modelle in Ollama registriert gefunden.")
        for m in models_info:
            name = m.get('model') if isinstance(m, dict) else getattr(m, 'model', str(m))
            size_bytes = m.get('size', 0) if isinstance(m, dict) else getattr(m, 'size', 0)
            if name:
                size_gb = size_bytes / (1024 ** 3) if size_bytes else 0.0
                local_models_dict[name] = size_gb
                print(f"  - {name} ({size_gb:.2f} GB)")
    except Exception as e:
        print(f"[Fehler beim Abrufen der Ollama-Modelle von {OLLAMA_HOST}]: {e}")
    print("\n Im Projektverzeichnis (data/models) sind diese Modelle vorhanden ")
    models_path = Path(project_models_dir)
    if not models_path.exists() and Path("data/models").exists():
        models_path = Path("data/models")
    if models_path.exists() and models_path.is_dir():
        gguf_files = list(models_path.rglob("*.gguf"))
        if gguf_files:
            for file_path in gguf_files:
                size_gb = file_path.stat().st_size / (1024 ** 3)
                print(f"  - {file_path.name} ({size_gb:.2f} GB)")
        else:
            print(f"  Keine .gguf-Modelldateien in '{models_path.resolve()}' gefunden.")
    else:
        print(f"  Verzeichnis '{project_models_dir}' wurde nicht gefunden.")
    return local_models_dict

def select_active_models(query: str) -> list:
    """
    Ermittelt den Schwerpunkt der Benutzeranfrage und wählt 
    die passenden Modelle direkt aus den globalen Variablen aus.
    """
    print('select_active_models')
    normalized_query = query.lower()
    is_code = any(kw in normalized_query for kw in ["code", "python", "script", "funktion", "bug", "programm"])
    is_logic = any(kw in normalized_query for kw in ["warum", "analysiere", "berechne", "logik", "vergleiche", "strategie"])
    if is_code:
        return [OLAMA_MODEL_CODE_LOW, OLAMA_MODEL_CODE_HIGH, OLAMA_MODEL_TEXT_LOW]
    elif is_logic:
        return [OLAMA_MODEL_LOGIC_LOW, OLAMA_MODEL_LOGIC_HIGH, OLAMA_MODEL_TEXT_LOW]
    else:
        return [OLAMA_MODEL_TEXT_LOW, OLAMA_MODEL_LOGIC_LOW, OLAMA_MODEL_TEXT_HIGH]

def validate_and_adjust_models(active_models: list, available_models_dict: dict) -> list:
    """
    Prüft, ob die gewünschten Modelle lokal vorhanden sind (unterstützt Teil- und 
    Toleranzmatching bei Namensabweichungen), bereinigt Leerzeichen/Zeilenumbrüche, 
    verhindert Duplikate und gleicht die Modellgrößen mit den Hardware-Ressourcen ab.
    """
    print('validate_and_adjust_models')
    validated_models = []
    if isinstance(available_models_dict, dict):
        available_names_cleaned = {str(m).strip(): size for m, size in available_models_dict.items()}
        available_names = list(available_names_cleaned.keys())
    else:
        available_names_cleaned = {str(m).strip(): 0.0 for m in available_models_dict}
        available_names = list(available_names_cleaned.keys())
        available_models_dict = available_names_cleaned
    print("\n Starte Validierung der aktiven Modelle ") 
    for model in active_models:
        clean_model = str(model).strip()
        matched_model = None
        if clean_model in available_names:
            matched_model = clean_model
        else:
            base_query = clean_model.split(':')[0].lower()
            for avail in available_names:
                if base_query in avail.lower():
                    matched_model = avail
                    print(f"[Info] Fuzzy-Match: '{clean_model}' wurde auf lokalem Modell '{matched_model}' abgebildet.")
                    break
        if matched_model:
            if matched_model not in validated_models:
                model_size = available_models_dict.get(matched_model, 0.0)
                if model_size > 70.0:  # Z.B. das 122B Modell
                    print(f"[Warnung] Modell '{matched_model}' ({model_size:.2f} GB) überschreitet die sicheren Hardware-Kapazitäten!")
                else:
                    validated_models.append(matched_model)
        else:
            print(f"[Warnung] Modell '{clean_model}' nicht auf Festplatte gefunden.")
            if available_names:
                fallback = next((m for m in available_names if m not in validated_models), available_names[0])
                print(f"-> Verwende stattdessen verfügbares Fallback-Modell: {fallback}")
                if fallback not in validated_models:
                    validated_models.append(fallback)
    print(f"Validierte aktive Modelle: {validated_models}\n")
    return validated_models

def check_hardware_capacity() -> bool:
    """
    Prüft den System-RAM und den kombinierten VRAM aller verfügbaren GPUs
    """
    print('check_hardware_capacity')
    mem = psutil.virtual_memory()
    free_ram_gb = mem.available / (1024 ** 3)
    total_ram_gb = mem.total / (1024 ** 3)
    print(f"[Hardware-Check] System RAM  - Gesamt: {total_ram_gb:.1f} GB | Frei: {free_ram_gb:.1f} GB")
    vram_sufficient = True
    if TORCH_AVAILABLE and torch.cuda.is_available():
        try:
            device_count = torch.cuda.device_count()
            total_free_vram = 0
            for i in range(device_count):
                free_bytes, _ = torch.cuda.mem_get_info(i)
                total_free_vram += free_bytes
            free_vram_gb = total_free_vram / (1024 ** 3)
            print(f"[Hardware-Check] GPU VRAM (Multi-GPU) - Freier VRAM gesamt: {free_vram_gb:.1f} GB ({device_count} GPUs aktiv)")
            if free_vram_gb < 6.0:
                print("[Hardware-Check] Warnung: VRAM-Kapazität kritisch (< 6 GB frei).")
                vram_sufficient = False
        except Exception as e:
            print(f"[Hardware-Check] VRAM-Abfrage fehlgeschlagen: {e}")
    else:
        print("[Hardware-Check] Keine CUDA-GPU via PyTorch erkannt. Prüfe rein über RAM.")
    if free_ram_gb < 4.0 or not vram_sufficient:
        print("[Hardware-Check] Entscheidung: Modus = SEQUENZIELL (Hardware-Schutz aktiv).")
        return False
    else:
        print("[Hardware-Check] Entscheidung: Modus = PARALLEL (Ausreichend Ressourcen).")
        return True

async def query_single_model(model_name: str, query: str) -> dict:
    """
    Führt die Anfrage für ein einzelnes Ollama-Modell aus (direkt & sachlich ohne Smalltalk).
    """
    print('query_single_model')
    system_prompt = (
        "Du bist ein präziser Fachassistent. Antworte direkt, sachlich und auf den Punkt. "
        "Vermeide jeglichen Smalltalk, Begrüßungen oder ausschweifende Einleitungen."
    )
    try:
        response = await asyncio.to_thread(
            ollama.chat,
            model=model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': query}
            ],
            options={'temperature': 0.1}
        )
        answer = response['message']['content']
        return {
            "model": model_name,
            "answer": answer,
            "status": "success",
            "token_count": len(answer.split())
        }
    except Exception as e:
        return {
            "model": model_name,
            "answer": f"Fehler bei der Ausführung: {str(e)}",
            "status": "error",
            "token_count": 0
        }

async def execute_model_queries(models: list, query: str) -> list:
    """
    Steuert die Ausführung basierend auf der kombinierten RAM- und VRAM-Prüfung.
    """
    print('execute_model_queries')
    is_hardware_capable = check_hardware_capacity()
    results = []
    if is_hardware_capable:
        tasks = [query_single_model(model, query) for model in models]
        results = await asyncio.gather(*tasks)
    else:
        for model in models:
            print(f"Führe Modell sequenziell aus: {model}...")
            res = await query_single_model(model, query)
            results.append(res)
    return results

In [4]:
# Pipeline-Ausführung
raw_query = "Wie wird das Wetter heute wohl in unserem Landkreis?"
cleaned_query = clean_query(raw_query, language='german', additional_stopwords=["wohl"], exclude_stopwords=["wetter"])

# Wunsch-Modelle anhand der Anfrage ermitteln
initial_active_models = select_active_models(cleaned_query)

# Gegen lokale Festplatte validieren (gegen 404-Fehler)
local_models = get_available_local_models()
ACTIVE_MODELS = validate_and_adjust_models(initial_active_models, local_models)

print(f"Originale Anfrage : {raw_query}")
print(f"Bereinigte Anfrage: {cleaned_query}")
print(f"Validierte aktive Modelle: {ACTIVE_MODELS}\n")

# Start der Abfragen mit Hardware-Überwachung
evaluation_results = await execute_model_queries(ACTIVE_MODELS, cleaned_query)

# Ergebnis-Vorschau
for res in evaluation_results:
    print(f"\n Modell: {res['model']} ({res['status']}) ")
    print(res['answer'][:300] + "..." if len(res['answer']) > 300 else res['answer'])

clean_query
select_active_models
get_available_local_models

 Registrierte Ollama-Modelle (über API: http://127.0.0.1:11434) 
  - qwen3.6:27b (16.55 GB)
  - codestral:22b (11.71 GB)
  - qwen3.5:122b (75.78 GB)
  - gemma3:4b (3.12 GB)
  - moondream:latest (1.62 GB)
  - deepseek-r1-0528-qwen3-8b-bf16:latest (15.26 GB)
  - qwen3-coder-next-ud-q4-k-xl:latest (46.20 GB)

 Im Projektverzeichnis (data/models) sind diese Modelle vorhanden 
  - DeepSeek-R1-0528-Qwen3-8B-BF16.gguf (15.26 GB)
  - Qwen3-Coder-Next-UD-Q4_K_XL.gguf (46.20 GB)
  - Qwen3.5-122B-A10B-Uncensored-HauhauCS-Aggressive-Q4_K_P.gguf (73.28 GB)
validate_and_adjust_models

 Starte Validierung der aktiven Modelle 
[Info] Fuzzy-Match: 'deepseek-r1:8b' wurde auf lokalem Modell 'deepseek-r1-0528-qwen3-8b-bf16:latest' abgebildet.
Validierte aktive Modelle: ['gemma3:4b', 'deepseek-r1-0528-qwen3-8b-bf16:latest', 'qwen3.6:27b']

Originale Anfrage : Wie wird das Wetter heute wohl in unserem Landkreis?
Bereinigte Anfrage: Wetter heute La